# Backend scaling

Point cloud at `N = 100_000`. Always benchmarks `local`. If
`ZV_BENCH_S3_URL` is set in the env (and `obstore` / `fsspec` are
installed), also benchmarks those against the URL.

Set up a cloud target with:

```bash
export ZV_BENCH_S3_URL="s3://my-bucket/zv-bench/"
```

When `ZV_BENCH_S3_URL` is unset, the notebook reports a one-row
local-only result.

In [ ]:
import os, time, tempfile, shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


def _time(fn, *args, **kwargs):
    """Call fn(*args, **kwargs); return (elapsed_seconds, result)."""
    t0 = time.perf_counter()
    out = fn(*args, **kwargs)
    return time.perf_counter() - t0, out


def _store_bytes(path):
    """Total on-disk size of a store directory, in bytes."""
    p = Path(path)
    return sum(f.stat().st_size for f in p.rglob('*') if f.is_file())


def _new_store(prefix):
    """Fresh tempdir + zarrvectors path."""
    return Path(tempfile.mkdtemp(prefix=f'zvbench_{prefix}_')) / 'store.zarrvectors'

## 1 · Setup + backend availability

In [ ]:
from zarr_vectors.types.points import write_points, read_points
from zarr_vectors.exceptions import StoreError

N = 100_000
CHUNK = (200.0, 200.0, 200.0)
BIN   = (50.0, 50.0, 50.0)
SEED  = 0

cloud_url = os.environ.get('ZV_BENCH_S3_URL')
print(f'local backend:  always benchmarked')
print(f'cloud backends: {"benchmarked against " + cloud_url if cloud_url else "SKIPPED (set ZV_BENCH_S3_URL to enable)"}')

## 2 · Synthetic input (shared across backends)

In [ ]:
rng = np.random.default_rng(SEED)
positions = rng.uniform(0, 1000, (N, 3)).astype(np.float32)
intensity = rng.uniform(0, 1, N).astype(np.float32)

## 3 · Run the sweep

In [ ]:
def run_one(label, target, backend):
    """Time write + read against `target` using `backend`. Returns row dict."""
    try:
        tw, _ = _time(
            write_points, target, positions,
            chunk_shape=CHUNK, bin_shape=BIN,
            attributes={'intensity': intensity},
            backend=backend,
        )
        tr, _ = _time(read_points, target, attribute_names=['intensity'], backend=backend)
        size_MB = _store_bytes(target) / 1e6 if backend == 'local' else float('nan')
        return {
            'backend': label,
            'write_s': round(tw, 3),
            'read_s':  round(tr, 3),
            'size_MB': round(size_MB, 2) if size_MB == size_MB else None,  # nan check
        }
    except (ImportError, StoreError) as e:
        return {'backend': label, 'write_s': None, 'read_s': None, 'size_MB': None, 'skipped': str(e)[:60]}


rows = []

# Local (always)
local_store = _new_store('backend_local')
rows.append(run_one('local', str(local_store), 'local'))
shutil.rmtree(Path(local_store).parent, ignore_errors=True)

# obstore / fsspec (only if cloud_url set)
if cloud_url:
    for backend in ('obstore', 'fsspec'):
        # Use a per-backend subpath so runs don't collide.
        target = cloud_url.rstrip('/') + f'/run_{backend}'
        rows.append(run_one(backend, target, backend))

df = pd.DataFrame(rows)

## 4 · Results

In [ ]:
df

## 5 · Plot

In [ ]:
plot_df = df.dropna(subset=['write_s', 'read_s'])
if not plot_df.empty:
    fig, ax = plt.subplots(figsize=(5, 3.5))
    x = np.arange(len(plot_df))
    ax.bar(x - 0.18, plot_df['write_s'], width=0.36, label='write (s)')
    ax.bar(x + 0.18, plot_df['read_s'],  width=0.36, label='read (s)')
    ax.set_xticks(x)
    ax.set_xticklabels(plot_df['backend'])
    ax.set_ylabel('seconds')
    ax.set_title(f'Write / read time per backend (N = {N:,})')
    ax.legend()
    ax.grid(True, axis='y', alpha=0.3)
    plt.tight_layout()
else:
    print('no successful backends to plot')